# Military Spending Analysis — Data Insights & Visualization & Prediction


**Data source:** Stockholm International Peace Research Institute (SIPRI), 2026, via Our World in Data  
**Coverage:** 1988–2025 · Constant 2024 USD · Top 7 spenders (ever)  
**Countries tracked:** China, France, Germany, India, Italy, Japan, Russia, Saudi Arabia, USSR, United Kingdom, United States

---

This notebook covers three analytical dimensions:
1. **Comparison Table** — How much did each country's military spending change from start to end year?
2. **Events Table** — When did one country overtake another in the top-7 rankings?
3. **Growth Summary** — Which country had the fastest growth rate each year?

Each section contains: data loading → key insights → interactive HTML chart generation.


## 0. Setup

In [14]:
import pandas as pd
import json
import os

# Output directory for HTML charts
os.makedirs('charts', exist_ok=True)
print("Ready.")


Ready.


---
## 1. Comparison Table — Start vs End Year Spending

### What this data shows
Each country is compared between its earliest meaningful data point and 2025.  
Special rules apply:
- **USSR**: 1988 vs 1990 (last year before dissolution)
- **Russia**: 1992 vs 2025 (first year as independent state)
- **China**: 1989 vs 2025 (earliest available data)
- **All others**: 1988 vs 2025


In [15]:
df_comp = pd.read_csv('comparison_table.csv')
print(df_comp.to_string(index=False))


       country  start_year  start_spending_B  end_year  end_spending_B  change_B change_pct                                                          note
         China        1989             19.91      2025          335.02    315.11   +1582.7% No data in 1988, using earliest available year (1989) vs 2025
        France        1988             56.68      2025           64.52      7.84     +13.8%                                                  1988 vs 2025
       Germany        1988             67.19      2025          106.73     39.54     +58.8%                                                  1988 vs 2025
         India        1988             21.06      2025           93.25     72.19    +342.8%                                                  1988 vs 2025
         Italy        1988             33.22      2025           45.44     12.22     +36.8%                                                  1988 vs 2025
         Japan        1988             30.47      2025           59.49     2

### Key Insights

#### 🔴 China: +1,583% — by far the largest relative increase
- 1989: **$19.9B** → 2025: **$335.0B**
- Grew at a sustained pace for 36 consecutive years with no single year of decline
- In 1989, China spent less than India. By 2025, it spends **3.6× more than India**
- Still only **36%** of US spending — the gap remains enormous in absolute terms

#### 🟠 India: +343% — the stealth climber
- 1988: **$21.1B** → 2025: **$93.3B**
- Entered the top 7 in 2004 and has stayed there
- Overtook France (2009), Saudi Arabia (2019) along the way
- Now the **5th largest spender globally**

#### 🔵 Russia: +233% from a collapsed base
- 1992 (post-USSR): **$47.5B** → 2025: **$158.2B**
- The 1992 starting point is deceptive — USSR spent **$282.9B** in 1988
- Russia's 2025 spending ($158B) is still only **56% of what the USSR spent in 1988**
- The 2022 Ukraine invasion caused a **+29.5% spike** in a single year

#### 🟡 Saudi Arabia: +204% — regional tensions + arms procurement
- Spending reflects sustained investment in defence modernisation and strategic posture across the Gulf region
- Reached peak levels in 2015–2016, coinciding with heightened regional tensions
- Entered the top 3 in 2013, then fell back — ranking has fluctuated heavily amid shifting strategic priorities

#### ⚪ Western powers: near-stagnation
- **United States**: +13.1% over 37 years (barely above inflation)
- **France**: +13.8%
- **United Kingdom**: +18.2%
- These countries maintained dominance through existing scale, not growth

#### 🔴 USSR: −21.6% in just 3 years (1988–1990)
- Already contracting before the 1991 dissolution
- Went from **$282.9B** (1988) to **$221.9B** (1990)
- A year later, the entire entity ceased to exist


In [16]:
df_sorted = df_comp.copy()
df_sorted['pct_numeric'] = df_sorted['change_pct'].str.replace('%','').str.replace('+','').astype(float)
df_sorted = df_sorted.sort_values('pct_numeric', ascending=False)

print("Ranking by % change:")
for _, row in df_sorted.iterrows():
    bar = '█' * max(1, int(abs(row['pct_numeric']) / 80))
    sign = '+' if row['pct_numeric'] >= 0 else ''
    print(f"  {row['country']:<16} {sign}{row['pct_numeric']:>8.1f}%  {bar}")


Ranking by % change:
  China            +  1582.7%  ███████████████████
  India            +   342.8%  ████
  Russia           +   233.4%  ██
  Saudi Arabia     +   204.0%  ██
  Japan            +    95.2%  █
  Germany          +    58.8%  █
  Italy            +    36.8%  █
  United Kingdom   +    18.2%  █
  France           +    13.8%  █
  United States    +    13.1%  █
  USSR                -21.6%  █


### Generate Chart 1 — Diverging Bar Chart

In [17]:
html_chart1 = r'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Military Spending — Change by Country</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
<style>
  * { margin:0; padding:0; box-sizing:border-box; }
  body { background:#0d0d0d; font-family:Georgia,serif; color:#E8E4DC; min-height:100vh; display:flex; align-items:center; justify-content:center; padding:40px 24px; }
  .card { background:white; border:0.5px solid #2a2a2a; border-radius:12px; padding:36px 40px; width:100%; max-width:780px; }
  .header { margin-bottom:28px; }
  .header h1 { font-size:18px; font-weight:normal; letter-spacing:0.04em; color:black; margin-bottom:6px; }
  .header p { font-size:12px; color:black; font-family:'Courier New',monospace; }
  .chart-wrap { position:relative; width:100%; height:480px; }
  .notes { margin-top:20px; display:flex; flex-direction:column; gap:7px; }
  .note { display:flex; gap:10px; font-size:11px; color:black; font-family:'Courier New',monospace; line-height:1.5; }
  .note-num { color:#3a3a3a; flex-shrink:0; }
  .note em { color:black; font-style:normal; }
  .source { margin-top:24px; padding-top:16px; border-top:0.5px solid #1e1e1e; font-size:10px; color:#3a3a3a; font-family:'Courier New',monospace; }
</style>
</head>
<body>
<div class="card">
  <div class="header">
    <h1>Military spending — change from start to end year</h1>
    <p>SIPRI Military Expenditure Database · constant 2024 USD · 1988–2025</p>
  </div>
  <div class="chart-wrap">
    <canvas id="compChart" role="img" aria-label="Diverging horizontal bar chart of military spending percentage change by country.">Military spending change by country.</canvas>
  </div>
  <div class="notes">
    <div class="note"><span class="note-num">①</span><span>Hover over each bar for start year, end year, spending values, and absolute change.</span></div>
    <div class="note"><span class="note-num">②</span><span><em>USSR: 1988–1990 only</em> (dissolved 1991). <em>Russia: 1992–2025</em> (from independence). <em>China: 1989–2025</em> (earliest available data).</span></div>
  </div>
  <div class="source">Data: Stockholm International Peace Research Institute (SIPRI), 2026, via Our World in Data</div>
</div>
<script>
const data = [
  {country:'China',startYear:1989,startB:19.91,endYear:2025,endB:335.02,pct:1582.7},
  {country:'India',startYear:1988,startB:21.06,endYear:2025,endB:93.25,pct:342.8},
  {country:'Russia',startYear:1992,startB:47.46,endYear:2025,endB:158.24,pct:233.4},
  {country:'Saudi Arabia',startYear:1988,startB:26.80,endYear:2025,endB:81.47,pct:204.0},
  {country:'Japan',startYear:1988,startB:30.47,endYear:2025,endB:59.49,pct:95.2},
  {country:'Germany',startYear:1988,startB:67.19,endYear:2025,endB:106.73,pct:58.8},
  {country:'Italy',startYear:1988,startB:33.22,endYear:2025,endB:45.44,pct:36.8},
  {country:'United Kingdom',startYear:1988,startB:74.43,endYear:2025,endB:87.98,pct:18.2},
  {country:'France',startYear:1988,startB:56.68,endYear:2025,endB:64.52,pct:13.8},
  {country:'United States',startYear:1988,startB:821.40,endYear:2025,endB:929.16,pct:13.1},
  {country:'USSR',startYear:1988,startB:282.92,endYear:1990,endB:221.93,pct:-21.6},
];
new Chart(document.getElementById('compChart'),{
  type:'bar',
  data:{labels:data.map(d=>d.country),datasets:[{data:data.map(d=>d.pct),backgroundColor:data.map(d=>d.pct<0?'#7a2020':'#2a5a8a'),hoverBackgroundColor:data.map(d=>d.pct<0?'#c04040':'#3a7abf'),borderRadius:3,borderSkipped:false}]},
  options:{indexAxis:'y',responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false},tooltip:{callbacks:{title:ctx=>data[ctx[0].dataIndex].country,label:ctx=>{const d=data[ctx.dataIndex];const changeB=(d.endB-d.startB).toFixed(1);const sign=d.endB>=d.startB?'+':'';const pctStr=(d.pct>=0?'+':'')+d.pct.toFixed(1)+'%';return['Start:   '+d.startYear+'  →  $'+d.startB.toFixed(1)+'B','End:     '+d.endYear+'  →  $'+d.endB.toFixed(1)+'B','──────────────────────','Change:  '+sign+'$'+changeB+'B  ('+pctStr+')'];}},backgroundColor:'rgba(10,10,10,0.95)',titleColor:'#ffffff',bodyColor:'#ffffff',padding:12,displayColors:false,borderColor:'#2a2a2a',borderWidth:1}},scales:{x:{grid:{color:'rgba(255,255,255,0.05)'},ticks:{color:'#444',font:{size:11,family:'Courier New'},callback:v=>(v>=0?'+':'')+v+'%'},title:{display:true,text:'Change in military spending from start to end year (%)',color:'#444',font:{size:11,family:'Courier New'}}},y:{grid:{display:false},ticks:{color:'#888',font:{size:12,family:'Georgia'}}}}}
});
</script>
</body>
</html>'''

with open('charts/chart1_comparison.html', 'w') as f:
    f.write(html_chart1)
print("✅ charts/chart1_comparison.html saved")


✅ charts/chart1_comparison.html saved


---
## 2. Events Table -- Ranking Overtake Events

### What this data shows
Every year where one country surpassed another among countries that ever entered the top-7 military spenders.
63 overtake events detected between 1988 and 2025.




In [18]:
df_events = pd.read_csv('events_table.csv')
print(f"Total overtake events: {len(df_events)}")
print()
print(df_events.to_string(index=False))


Total overtake events: 63

 year      surpasser      surpassed  surpasser_rank  surpassed_rank  surpasser_spending_B  surpassed_spending_B  gap_B
 1990          Japan          Italy               6               7                 33.35                 32.21   1.14
 1991          China          India               8               9                 23.00                 20.41   2.59
 1994         France        Germany               3               4                 55.35                 52.11   3.24
 1995          China   Saudi Arabia               8               9                 25.71                 23.00   2.71
 1995          Italy         Russia               6               7                 28.41                 25.81   2.60
 1995          Japan         Russia               5               7                 35.12                 25.81   9.31
 1996          China         Russia               7               8                 27.23                 24.37   2.86
 1996          India 

### Key Insights

#### Most turbulent years
| Year | Events | Why |
|------|--------|-----|
| **2002** | 4 events | China overtook France; Italy, Japan regained ground over Saudi Arabia |
| **2022** | 4 events | Russia Ukraine invasion spike; Saudi Arabia also climbed over UK |
| **2024** | 4 events | Germany and UK NATO-driven surge; both overtook India and Saudi Arabia |
| **1995** | 3 events | Russia post-Soviet collapse hit bottom; Italy, Japan, China all passed it |
| **1997** | 3 events | Saudi Arabia geopolitical surge -- overtook China, India, Russia simultaneously |
| **1999** | 3 events | China WTO period surge -- overtook Italy, Japan, Saudi Arabia in one year |
| **2017** | 3 events | Russia crashed after oil/sanctions; India, Saudi Arabia, UK all leapfrogged it |

#### China's methodical climb (4 events, all upward, never reversed)
- **1999**: Overtook Italy, Japan, Saudi Arabia simultaneously -- rank 9 to rank 5 in one year
- **2001**: Overtook Germany -- entered top 4
- **2002**: Overtook France -- entered top 3
- **2005**: Overtook UK -- reached **#2**, where it has remained ever since
- China has **never been overtaken** after reaching #2

#### India: the overlooked story (10 events as surpasser)
- Previously missed in v2 analysis due to a logic bug -- now fully captured
- **2004**: Overtook Italy and Japan simultaneously
- **2008**: Overtook Germany
- **2009**: Overtook France -- entered top 5
- **2016**: Overtook UK
- **2017**: Overtook Russia -- entered top 4
- **2019**: Overtook Saudi Arabia -- reached **#3**

#### Russia: most volatile (appears in 20+ events -- both directions)
- Collapsed from USSR-era dominance to rank 10 by late 1990s
- Climbed back through 2000s oil boom
- Crashed again after 2014 sanctions and oil price collapse
- Surged back to #3 in 2022 driven by Ukraine war spending

#### Saudi Arabia: geopolitics and arms procurement driven
- Spending reflects sustained regional tensions and large-scale modern weapons procurement
- In 1997 alone overtook China, India, and Russia simultaneously
- Peaked at #3 in 2013 during major defence modernisation programme
- Highly volatile: has been both surpasser and surpassed multiple times with the same countries

#### Countries that never overtook anyone
- **United States**: #1 the entire period
- **USSR**: dissolved before being overtaken


In [19]:
# Summary: who overtook whom the most
print("Times each country overtook others:")
print(df_events['surpasser'].value_counts().to_string())
print()
print("Times each country was overtaken:")
print(df_events['surpassed'].value_counts().to_string())
print()
print("Events per year -- ranked by number of events:")
events_per_year = df_events.groupby('year').size().sort_values(ascending=False)
print(events_per_year.to_string())
print()
print("Years with NO overtake events (rankings were stable):")
all_years = list(range(1989, 2026))
years_with_events = df_events['year'].unique()
empty = sorted([y for y in all_years if y not in years_with_events])
print(empty)


Times each country overtook others:
surpasser
Saudi Arabia      14
India             11
Russia            10
China              9
Germany            5
Japan              4
Italy              4
United Kingdom     4
France             2

Times each country was overtaken:
surpassed
Saudi Arabia      12
Italy              8
Russia             8
United Kingdom     8
India              7
Japan              7
Germany            6
France             6
China              1

Events per year -- ranked by number of events:
year
2002    4
2022    4
2024    4
2008    3
2006    3
2005    3
2004    3
2001    3
2017    3
1999    3
1997    3
1995    3
2016    2
1998    2
1996    2
2021    2
2009    2
2019    2
2025    2
2020    1
1990    1
2014    1
2013    1
2012    1
2011    1
1991    1
2000    1
1994    1
2007    1

Years with NO overtake events (rankings were stable):
[1989, 1992, 1993, 2003, 2010, 2015, 2018, 2023]


### Generate Chart 2 — Timeline Scatter Plot

In [1]:
html_chart2 = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Military Spending - Ranking Overtake Events</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
<style>
  *{margin:0;padding:0;box-sizing:border-box;}
  body{background:black;font-family:Georgia,serif;color:black;min-height:100vh;display:flex;align-items:center;justify-content:center;padding:40px 24px;}
  .card{background:white;border:0.5px solid #ddd;border-radius:12px;padding:36px 40px;width:100%;max-width:860px;}
  .header{margin-bottom:20px;}
  .header h1{font-size:18px;font-weight:normal;letter-spacing:0.04em;color:black;margin-bottom:6px;}
  .header p{font-size:12px;color:black;font-family:'Courier New',monospace;}
  .filter-wrap{display:flex;flex-wrap:wrap;align-items:center;gap:8px;margin-bottom:16px;}
  .filter-label{font-size:11px;color:black;font-family:'Courier New',monospace;flex-shrink:0;}
  .filter-btn{padding:4px 12px;border-radius:16px;border:1px solid #ccc;background:white;color:black;font-size:11px;font-family:'Courier New',monospace;cursor:pointer;transition:all 0.15s;}
  .filter-btn:hover{background:#f0f0f0;border-color:#999;}
  .filter-btn.active{background:black;color:white;border-color:black;}
  .legend{display:flex;flex-wrap:wrap;gap:12px;margin-bottom:8px;font-size:11px;color:black;font-family:'Courier New',monospace;}
  .legend-item{display:flex;align-items:center;gap:5px;}
  .legend-dot{width:9px;height:9px;border-radius:50%;flex-shrink:0;}
  .legend-note{width:100%;font-size:10px;color:#888;font-family:'Courier New',monospace;margin-top:2px;}
  .event-count{font-size:11px;color:#666;font-family:'Courier New',monospace;margin-bottom:8px;}
  .event-count span{font-weight:bold;color:black;}
  .chart-wrap{position:relative;width:100%;height:400px;}
  .notes{margin-top:20px;display:flex;flex-direction:column;gap:7px;}
  .note{display:flex;gap:10px;font-size:11px;color:black;font-family:'Courier New',monospace;line-height:1.5;}
  .note-num{color:#666;flex-shrink:0;}
  .source{margin-top:24px;padding-top:16px;border-top:0.5px solid #ddd;font-size:10px;color:#666;font-family:'Courier New',monospace;}
</style>
</head>
<body>
<div class="card">
  <div class="header">
    <h1>Ranking overtake events -- top 7 military spenders</h1>
    <p>SIPRI Military Expenditure Database · constant 2024 USD · 1988-2025 · 63 events total</p>
  </div>

  <div class="filter-wrap">
    <span class="filter-label">Filter by country:</span>
    <button class="filter-btn active" data-country="All" onclick="setFilter('All')">All countries</button>
    <button class="filter-btn" data-country="China" onclick="setFilter('China')">China</button>
    <button class="filter-btn" data-country="Russia" onclick="setFilter('Russia')">Russia</button>
    <button class="filter-btn" data-country="India" onclick="setFilter('India')">India</button>
    <button class="filter-btn" data-country="Saudi Arabia" onclick="setFilter('Saudi Arabia')">Saudi Arabia</button>
    <button class="filter-btn" data-country="Germany" onclick="setFilter('Germany')">Germany</button>
    <button class="filter-btn" data-country="United Kingdom" onclick="setFilter('United Kingdom')">United Kingdom</button>
    <button class="filter-btn" data-country="France" onclick="setFilter('France')">France</button>
    <button class="filter-btn" data-country="Italy" onclick="setFilter('Italy')">Italy</button>
    <button class="filter-btn" data-country="Japan" onclick="setFilter('Japan')">Japan</button>
  </div>

  <div class="legend">
    <div class="legend-item"><div class="legend-dot" style="background:#e05c2a;"></div>China</div>
    <div class="legend-item"><div class="legend-dot" style="background:#3a7abf;"></div>Russia</div>
    <div class="legend-item"><div class="legend-dot" style="background:#8050b0;"></div>India</div>
    <div class="legend-item"><div class="legend-dot" style="background:#d4a020;"></div>Saudi Arabia</div>
    <div class="legend-item"><div class="legend-dot" style="background:#50a060;"></div>Germany</div>
    <div class="legend-item"><div class="legend-dot" style="background:#2a7a6a;"></div>United Kingdom</div>
    <div class="legend-item"><div class="legend-dot" style="background:#888;"></div>Others</div>
    <div class="legend-note">Color = surpassing country. Dot size reflects spending gap at time of overtake.</div>
  </div>

  <div class="event-count" id="eventCount">Showing <span>63</span> events</div>

  <div class="chart-wrap">
    <canvas id="eventsChart" role="img" aria-label="Scatter plot of 63 military spending ranking overtake events 1988-2025.">63 ranking overtake events plotted by year and rank achieved.</canvas>
  </div>

  <div class="notes">
    <div class="note"><span class="note-num">1</span><span>Each dot = one overtake event. Y-axis = rank achieved by the surpasser after overtaking. Color = surpassing country. Hover for full details.</span></div>
    <div class="note"><span class="note-num">2</span><span>Use the filter buttons above to show only events involving a specific country -- as surpasser or surpassed.</span></div>
    <div class="note"><span class="note-num">3</span><span>Clustered years (1999, 2017, 2024) indicate periods of intense reshuffling in global military rankings.</span></div>
  </div>

  <div class="source">Data: Stockholm International Peace Research Institute (SIPRI), 2026, via Our World in Data</div>
</div>

<script>
const allEvents=[
  {year:1990,surpasser:'Japan',surpassed:'Italy',rank:6,surpasserB:33.35,surpassedB:32.21,gapB:1.14},
  {year:1991,surpasser:'China',surpassed:'India',rank:8,surpasserB:23.0,surpassedB:20.41,gapB:2.59},
  {year:1994,surpasser:'France',surpassed:'Germany',rank:3,surpasserB:55.35,surpassedB:52.11,gapB:3.24},
  {year:1995,surpasser:'China',surpassed:'Saudi Arabia',rank:8,surpasserB:25.71,surpassedB:23.0,gapB:2.71},
  {year:1995,surpasser:'Italy',surpassed:'Russia',rank:6,surpasserB:28.41,surpassedB:25.81,gapB:2.6},
  {year:1995,surpasser:'Japan',surpassed:'Russia',rank:5,surpasserB:35.12,surpassedB:25.81,gapB:9.31},
  {year:1996,surpasser:'China',surpassed:'Russia',rank:7,surpasserB:27.23,surpassedB:24.37,gapB:2.86},
  {year:1996,surpasser:'India',surpassed:'Saudi Arabia',rank:9,surpasserB:23.21,surpassedB:22.96,gapB:0.25},
  {year:1997,surpasser:'Saudi Arabia',surpassed:'China',rank:7,surpasserB:31.18,surpassedB:29.05,gapB:2.13},
  {year:1997,surpasser:'Saudi Arabia',surpassed:'India',rank:7,surpasserB:31.18,surpassedB:25.69,gapB:5.49},
  {year:1997,surpasser:'Saudi Arabia',surpassed:'Russia',rank:7,surpasserB:31.18,surpassedB:26.64,gapB:4.54},
  {year:1998,surpasser:'India',surpassed:'Russia',rank:9,surpasserB:26.81,surpassedB:15.84,gapB:10.97},
  {year:1998,surpasser:'Saudi Arabia',surpassed:'Italy',rank:6,surpasserB:36.02,surpassedB:33.91,gapB:2.11},
  {year:1999,surpasser:'China',surpassed:'Italy',rank:5,surpasserB:38.67,surpassedB:35.23,gapB:3.44},
  {year:1999,surpasser:'China',surpassed:'Japan',rank:5,surpasserB:38.67,surpassedB:35.92,gapB:2.75},
  {year:1999,surpasser:'China',surpassed:'Saudi Arabia',rank:5,surpasserB:38.67,surpassedB:32.06,gapB:6.61},
  {year:2000,surpasser:'Italy',surpassed:'Japan',rank:6,surpasserB:37.58,surpassedB:36.11,gapB:1.47},
  {year:2001,surpasser:'China',surpassed:'Germany',rank:4,surpasserB:49.63,surpassedB:47.96,gapB:1.67},
  {year:2001,surpasser:'Saudi Arabia',surpassed:'Italy',rank:6,surpasserB:37.64,surpassedB:36.96,gapB:0.68},
  {year:2001,surpasser:'Saudi Arabia',surpassed:'Japan',rank:6,surpasserB:37.64,surpassedB:36.74,gapB:0.9},
  {year:2002,surpasser:'China',surpassed:'France',rank:3,surpasserB:57.01,surpassedB:50.89,gapB:6.12},
  {year:2002,surpasser:'India',surpassed:'Saudi Arabia',rank:8,surpasserB:33.16,surpassedB:33.04,gapB:0.12},
  {year:2002,surpasser:'Italy',surpassed:'Saudi Arabia',rank:6,surpasserB:37.97,surpassedB:33.04,gapB:4.93},
  {year:2002,surpasser:'Japan',surpassed:'Saudi Arabia',rank:7,surpasserB:36.92,surpassedB:33.04,gapB:3.88},
  {year:2004,surpasser:'India',surpassed:'Italy',rank:6,surpasserB:39.38,surpassedB:38.41,gapB:0.97},
  {year:2004,surpasser:'India',surpassed:'Japan',rank:6,surpasserB:39.38,surpassedB:36.82,gapB:2.56},
  {year:2004,surpasser:'Saudi Arabia',surpassed:'Japan',rank:8,surpasserB:36.92,surpassedB:36.82,gapB:0.1},
  {year:2005,surpasser:'China',surpassed:'United Kingdom',rank:2,surpasserB:74.62,surpassedB:72.5,gapB:2.12},
  {year:2005,surpasser:'Saudi Arabia',surpassed:'India',rank:6,surpasserB:44.58,surpassedB:41.91,gapB:2.67},
  {year:2005,surpasser:'Saudi Arabia',surpassed:'Italy',rank:6,surpasserB:44.58,surpassedB:36.95,gapB:7.63},
  {year:2006,surpasser:'Russia',surpassed:'Italy',rank:8,surpasserB:39.17,surpassedB:35.76,gapB:3.41},
  {year:2006,surpasser:'Russia',surpassed:'Japan',rank:8,surpasserB:39.17,surpassedB:36.29,gapB:2.88},
  {year:2006,surpasser:'Saudi Arabia',surpassed:'Germany',rank:5,surpasserB:50.79,surpassedB:44.46,gapB:6.33},
  {year:2007,surpasser:'Saudi Arabia',surpassed:'France',rank:4,surpasserB:58.5,surpassedB:52.98,gapB:5.52},
  {year:2008,surpasser:'India',surpassed:'Germany',rank:6,surpasserB:48.5,surpassedB:45.58,gapB:2.92},
  {year:2008,surpasser:'Italy',surpassed:'Japan',rank:9,surpasserB:36.17,surpassedB:35.47,gapB:0.7},
  {year:2008,surpasser:'Russia',surpassed:'Germany',rank:7,surpasserB:46.85,surpassedB:45.58,gapB:1.27},
  {year:2009,surpasser:'India',surpassed:'France',rank:5,surpasserB:57.1,surpassedB:56.48,gapB:0.62},
  {year:2009,surpasser:'Japan',surpassed:'Italy',rank:9,surpasserB:36.13,surpassedB:34.99,gapB:1.14},
  {year:2011,surpasser:'Russia',surpassed:'France',rank:6,surpasserB:53.53,surpassedB:52.21,gapB:1.32},
  {year:2012,surpasser:'Russia',surpassed:'India',rank:5,surpasserB:62.02,surpassedB:57.56,gapB:4.46},
  {year:2013,surpasser:'Saudi Arabia',surpassed:'United Kingdom',rank:3,surpasserB:80.73,surpassedB:70.6,gapB:10.13},
  {year:2014,surpasser:'Russia',surpassed:'United Kingdom',rank:4,surpasserB:69.71,surpassedB:69.34,gapB:0.37},
  {year:2016,surpasser:'India',surpassed:'United Kingdom',rank:5,surpasserB:67.2,surpassedB:66.44,gapB:0.76},
  {year:2016,surpasser:'Russia',surpassed:'Saudi Arabia',rank:3,surpasserB:80.55,surpassedB:72.62,gapB:7.93},
  {year:2017,surpasser:'India',surpassed:'Russia',rank:4,surpasserB:71.94,surpassedB:65.27,gapB:6.67},
  {year:2017,surpasser:'Saudi Arabia',surpassed:'Russia',rank:3,surpasserB:80.97,surpassedB:65.27,gapB:15.7},
  {year:2017,surpasser:'United Kingdom',surpassed:'Russia',rank:5,surpasserB:66.56,surpassedB:65.27,gapB:1.29},
  {year:2019,surpasser:'Germany',surpassed:'France',rank:7,surpasserB:56.7,surpassedB:56.07,gapB:0.63},
  {year:2019,surpasser:'India',surpassed:'Saudi Arabia',rank:3,surpasserB:79.73,surpassedB:74.25,gapB:5.48},
  {year:2020,surpasser:'United Kingdom',surpassed:'Saudi Arabia',rank:4,surpasserB:70.96,surpassedB:70.95,gapB:0.01},
  {year:2021,surpasser:'France',surpassed:'Germany',rank:7,surpasserB:60.51,surpassedB:59.89,gapB:0.62},
  {year:2021,surpasser:'Russia',surpassed:'Saudi Arabia',rank:5,surpasserB:68.5,surpassedB:67.38,gapB:1.12},
  {year:2022,surpasser:'Germany',surpassed:'France',rank:7,surpasserB:62.5,surpassedB:60.19,gapB:2.31},
  {year:2022,surpasser:'Russia',surpassed:'India',rank:3,surpasserB:88.68,surpassedB:83.22,gapB:5.46},
  {year:2022,surpasser:'Russia',surpassed:'United Kingdom',rank:3,surpasserB:88.68,surpassedB:73.19,gapB:15.49},
  {year:2022,surpasser:'Saudi Arabia',surpassed:'United Kingdom',rank:5,surpasserB:73.8,surpassedB:73.19,gapB:0.61},
  {year:2024,surpasser:'Germany',surpassed:'India',rank:5,surpasserB:86.15,surpassedB:85.6,gapB:0.55},
  {year:2024,surpasser:'Germany',surpassed:'Saudi Arabia',rank:5,surpasserB:86.15,surpassedB:80.33,gapB:5.82},
  {year:2024,surpasser:'United Kingdom',surpassed:'India',rank:4,surpasserB:89.81,surpassedB:85.6,gapB:4.21},
  {year:2024,surpasser:'United Kingdom',surpassed:'Saudi Arabia',rank:4,surpasserB:89.81,surpassedB:80.33,gapB:9.48},
  {year:2025,surpasser:'Germany',surpassed:'United Kingdom',rank:4,surpasserB:106.73,surpassedB:87.98,gapB:18.75},
  {year:2025,surpasser:'India',surpassed:'United Kingdom',rank:5,surpasserB:93.25,surpassedB:87.98,gapB:5.27}
];

const colorMap={'China':'#e05c2a','Russia':'#3a7abf','India':'#8050b0','Saudi Arabia':'#d4a020','Germany':'#50a060','United Kingdom':'#2a7a6a'};
const getColor=s=>colorMap[s]||'#888';
const maxGap=Math.max(...allEvents.map(e=>e.gapB));
let chart;

function filterEvents(country){
  return country==='All'?allEvents:allEvents.filter(e=>e.surpasser===country||e.surpassed===country);
}

function buildDatasets(filtered){
  const jitter={};
  const processed=filtered.map(e=>{
    const key=e.year+'-'+e.rank;
    jitter[key]=(jitter[key]||0);
    const j=jitter[key]*0.22;
    jitter[key]++;
    return{...e,_jitter:j};
  });
  return[{
    data:processed.map(e=>({x:e.year,y:e.rank+e._jitter})),
    backgroundColor:processed.map(e=>getColor(e.surpasser)),
    pointRadius:processed.map(e=>Math.max(7,Math.min(16,7+(e.gapB/maxGap)*9))),
    pointHoverRadius:processed.map(e=>Math.max(10,Math.min(20,10+(e.gapB/maxGap)*10))),
    _events:processed,
  }];
}

function setFilter(country){
  document.querySelectorAll('.filter-btn').forEach(btn=>btn.classList.toggle('active',btn.dataset.country===country));
  const filtered=filterEvents(country);
  const badge=document.getElementById('eventCount');
  badge.innerHTML=country==='All'
    ?`Showing <span>${filtered.length}</span> events`
    :`Showing <span>${filtered.length}</span> events involving <strong>${country}</strong>`;
  chart.data.datasets=buildDatasets(filtered);
  chart.update();
}

window.onload=function(){
  chart=new Chart(document.getElementById('eventsChart'),{
    type:'scatter',
    data:{datasets:buildDatasets(allEvents)},
    options:{
      responsive:true,maintainAspectRatio:false,
      plugins:{
        legend:{display:false},
        tooltip:{
          callbacks:{
            title:ctx=>{const e=chart.data.datasets[ctx[0].datasetIndex]._events[ctx[0].dataIndex];return'In '+e.year+':  '+e.surpasser+' overtook '+e.surpassed;},
            label:ctx=>{const e=chart.data.datasets[ctx.datasetIndex]._events[ctx.dataIndex];return['New rank:        #'+e.rank,e.surpasser+':   $'+e.surpasserB.toFixed(1)+'B',e.surpassed+':   $'+e.surpassedB.toFixed(1)+'B','Spending gap:    $'+e.gapB.toFixed(2)+'B'];}
          },
          backgroundColor:'rgba(10,10,10,0.92)',titleColor:'#ffffff',bodyColor:'#cccccc',padding:12,displayColors:false,borderColor:'#333',borderWidth:1,
        }
      },
      scales:{
        x:{min:1988,max:2026,grid:{color:'rgba(0,0,0,0.06)'},ticks:{color:'black',font:{size:10,family:'Courier New'},stepSize:1,autoSkip:false,maxRotation:45,callback:v=>Number.isInteger(v)?v:''}},
        y:{reverse:true,min:1.5,max:9.8,grid:{color:'rgba(0,0,0,0.08)'},border:{display:true},ticks:{color:'black',font:{size:12,family:'Courier New'},stepSize:1,padding:6,callback:v=>Number.isInteger(v)&&v>=2&&v<=9?'#'+v:''},title:{display:true,text:'Rank achieved by surpasser',color:'black',font:{size:11,family:'Courier New'}}}
      }
    }
  });
};
</script>
</body>
</html>
"""

with open('charts/chart2_events.html', 'w') as f:
    f.write(html_chart2)
print("charts/chart2_events.html saved")
print("v2 features: 63 events, country filter, dot size = spending gap, Y axis #2-#9")


charts/chart2_events.html saved
v2 features: 63 events, country filter, dot size = spending gap, Y axis #2-#9


---
## 3. Growth Summary — Fastest Growing Country per Year

### What this data shows
For each year, which country had the largest year-on-year percentage increase in military spending,  
and how did that compare to the absolute top spender (always the United States)?


In [21]:
df_growth = pd.read_csv('growth_summary.csv')
print(df_growth.to_string(index=False))


 year   top_spender  top_spending_B fastest_growing  fastest_growth_B  fastest_growth_pct
 1989 United States          814.46           Japan              1.32                 4.3
 1990 United States          780.58    Saudi Arabia              6.50                25.7
 1991 United States          689.54           China              1.36                 6.3
 1992 United States          726.64   United States             37.10                 5.4
 1993 United States          687.75           India              2.52                12.9
 1994 United States          652.00          France              0.29                 0.5
 1995 United States          609.03           China              0.96                 3.9
 1996 United States          575.90           Italy              2.89                10.2
 1997 United States          572.92    Saudi Arabia              8.22                35.8
 1998 United States          559.98    Saudi Arabia              4.84                15.5
 1999 Unit

### Key Insights

#### 🇺🇸 United States: perpetual #1, but rarely the fastest grower
- **Top spender every single year from 1988 to 2025** — without exception
- Was fastest grower in: 1992, 2000, 2002–2010, 2018–2020, 2023–2024
- The 2002–2009 streak reflects post-9/11 and Iraq/Afghanistan war spending
- Outside of war periods, US growth rate is modest (+2–7% per year)

#### 🇨🇳 China: fastest grower in 15 out of 37 years
- Dominated the 1990s growth race (1991, 1995, 1999, 2001)
- Returned as fastest grower in 2011–2013, 2015–2017, 2021, 2025
- Growth rate has **slowed** — from 20%+ in the 1990s to 6–8% in the 2020s
- But the absolute dollar increases keep getting larger as the base grows

#### 🇸🇦 Saudi Arabia: three explosive spikes
- **1997: +35.8%** — highest single-year rate in the entire dataset 
- **1998: +15.5%** — continued surge
- **2014: +17.9%** — Yemen War buildup begins
- Saudi Arabia’s defence spending closely tracks regional geopolitical tensions.

#### 🇷🇺 Russia: one dramatic spike
- **2022: +29.5%** — second highest rate in dataset, driven by Ukraine invasion
- Added $20.2B in a single year
- Before and after, Russia's growth is modest or negative

#### 📉 The "peace dividend" years (1989–1998)
- US spending actually **declined** from $821B (1988) to $560B (1998) — a 32% drop
- This is when China and India quietly accelerated their military build-up
- The world was demilitarizing while Asia was rearming

#### 🔎 Closest race to US dominance: never close
- Even in China's best year (+$23.1B in 2025), the US added more in absolute terms in most years
- The gap in absolute spending has never seriously narrowed


In [22]:
# Count fastest grower by country
print("Years as fastest grower by country:")
print(df_growth['fastest_growing'].value_counts().to_string())
print()

# Find years with biggest spikes
print("Top 10 highest single-year growth rates:")
top10 = df_growth.nlargest(10, 'fastest_growth_pct')[['year','fastest_growing','fastest_growth_pct','fastest_growth_B']]
print(top10.to_string(index=False))


Years as fastest grower by country:
fastest_growing
United States    16
China            12
Saudi Arabia      4
Japan             1
India             1
France            1
Italy             1
Russia            1

Top 10 highest single-year growth rates:
 year fastest_growing  fastest_growth_pct  fastest_growth_B
 1997    Saudi Arabia                35.8              8.22
 2022          Russia                29.5             20.18
 1990    Saudi Arabia                25.7              6.50
 1999           China                21.9              6.95
 2001           China                18.6              7.77
 2014    Saudi Arabia                17.9             14.42
 1998    Saudi Arabia                15.5              4.84
 2003   United States                13.8             91.19
 1993           India                12.9              2.52
 2002   United States                12.3             72.19


### Generate Chart 3 — Growth Rate Bar Chart

In [27]:
html_chart3 = r'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Military Spending — Fastest Growth Rate per Year</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
<style>
  *{margin:0;padding:0;box-sizing:border-box;}
  body{background:#0d0d0d;font-family:Georgia,serif;color:#E8E4DC;min-height:100vh;display:flex;align-items:center;justify-content:center;padding:40px 24px;}
  .card{background:white;border:0.5px solid #2a2a2a;border-radius:12px;padding:36px 40px;width:100%;max-width:860px;}
  .header{margin-bottom:20px;}
  .header h1{font-size:18px;font-weight:normal;letter-spacing:0.04em;color:black;margin-bottom:6px;}
  .header p{font-size:12px;color:black;font-family:'Courier New',monospace;}
  .legend{display:flex;flex-wrap:wrap;gap:12px;margin-bottom:16px;font-size:11px;color:black;font-family:'Courier New',monospace;}
  .legend-item{display:flex;align-items:center;gap:5px;}
  .legend-swatch{width:9px;height:9px;border-radius:2px;flex-shrink:0;}
  .chart-wrap{position:relative;width:100%;height:320px;}
  .notes{margin-top:20px;display:flex;flex-direction:column;gap:7px;}
  .note{display:flex;gap:10px;font-size:11px;color:black;font-family:'Courier New',monospace;line-height:1.5;}
  .note-num{color:#3a3a3a;flex-shrink:0;}
  .note em{color:black;font-style:normal;}
  .source{margin-top:24px;padding-top:16px;border-top:0.5px solid #1e1e1e;font-size:10px;color:#3a3a3a;font-family:'Courier New',monospace;}
</style>
</head>
<body>
<div class="card">
  <div class="header">
    <h1>Fastest military spending growth rate per year</h1>
    <p>SIPRI Military Expenditure Database · constant 2024 USD · 1989–2025</p>
  </div>
  <div class="legend">
    <div class="legend-item"><div class="legend-swatch" style="background:#808080;"></div>USA</div>
    <div class="legend-item"><div class="legend-swatch" style="background:#e05c2a;"></div>China</div>
    <div class="legend-item"><div class="legend-swatch" style="background:#3a7abf;"></div>Russia</div>
    <div class="legend-item"><div class="legend-swatch" style="background:#8050b0;"></div>India</div>
    <div class="legend-item"><div class="legend-swatch" style="background:#d4a020;"></div>Saudi Arabia</div>
    <div class="legend-item"><div class="legend-swatch" style="background:#50a060;"></div>Others</div>
  </div>
  <div class="chart-wrap">
    <canvas id="growthChart" role="img" aria-label="Bar chart of fastest military spending growth rate per year 1989-2025.">Fastest growing country growth rate per year 1989-2025.</canvas>
  </div>
  <div class="notes">
    <div class="note"><span class="note-num">①</span><span><em>United States was the #1 spender every single year from 1988 to 2025.</em> Bars show which country had the fastest growth rate that year, not the largest budget.</span></div>
    <div class="note"><span class="note-num">②</span><span>Hover over each bar for country name, growth rate, absolute increase, and US spending that year.</span></div>
  </div>
  <div class="source">Data: Stockholm International Peace Research Institute (SIPRI), 2026, via Our World in Data</div>
</div>
<script>
const rows=[
  {year:1989,fastest:'Japan',pct:4.3,growthB:1.32,topSpender:'United States',topB:814.46},
  {year:1990,fastest:'Saudi Arabia',pct:25.7,growthB:6.50,topSpender:'United States',topB:780.58},
  {year:1991,fastest:'China',pct:6.3,growthB:1.36,topSpender:'United States',topB:689.54},
  {year:1992,fastest:'United States',pct:5.4,growthB:37.10,topSpender:'United States',topB:726.64},
  {year:1993,fastest:'India',pct:12.9,growthB:2.52,topSpender:'United States',topB:687.75},
  {year:1994,fastest:'France',pct:0.5,growthB:0.29,topSpender:'United States',topB:652.00},
  {year:1995,fastest:'China',pct:3.9,growthB:0.96,topSpender:'United States',topB:609.03},
  {year:1996,fastest:'Italy',pct:10.2,growthB:2.89,topSpender:'United States',topB:575.90},
  {year:1997,fastest:'Saudi Arabia',pct:35.8,growthB:8.22,topSpender:'United States',topB:572.92},
  {year:1998,fastest:'Saudi Arabia',pct:15.5,growthB:4.84,topSpender:'United States',topB:559.98},
  {year:1999,fastest:'China',pct:21.9,growthB:6.95,topSpender:'United States',topB:561.36},
  {year:2000,fastest:'United States',pct:3.9,growthB:21.73,topSpender:'United States',topB:583.09},
  {year:2001,fastest:'China',pct:18.6,growthB:7.77,topSpender:'United States',topB:587.82},
  {year:2002,fastest:'United States',pct:12.3,growthB:72.19,topSpender:'United States',topB:660.01},
  {year:2003,fastest:'United States',pct:13.8,growthB:91.19,topSpender:'United States',topB:751.20},
  {year:2004,fastest:'United States',pct:9.0,growthB:67.55,topSpender:'United States',topB:818.75},
  {year:2005,fastest:'United States',pct:4.6,growthB:37.71,topSpender:'United States',topB:856.46},
  {year:2006,fastest:'United States',pct:1.4,growthB:12.34,topSpender:'United States',topB:868.80},
  {year:2007,fastest:'United States',pct:2.7,growthB:23.19,topSpender:'United States',topB:891.99},
  {year:2008,fastest:'United States',pct:7.3,growthB:64.88,topSpender:'United States',topB:956.87},
  {year:2009,fastest:'United States',pct:7.9,growthB:75.30,topSpender:'United States',topB:1032.17},
  {year:2010,fastest:'United States',pct:2.9,growthB:29.50,topSpender:'United States',topB:1061.67},
  {year:2011,fastest:'China',pct:7.4,growthB:9.67,topSpender:'United States',topB:1049.10},
  {year:2012,fastest:'China',pct:10.3,growthB:14.50,topSpender:'United States',topB:990.83},
  {year:2013,fastest:'China',pct:8.1,growthB:12.66,topSpender:'United States',topB:914.62},
  {year:2014,fastest:'Saudi Arabia',pct:17.9,growthB:14.42,topSpender:'United States',topB:858.36},
  {year:2015,fastest:'China',pct:7.9,growthB:14.28,topSpender:'United States',topB:838.87},
  {year:2016,fastest:'China',pct:5.8,growthB:11.29,topSpender:'United States',topB:836.29},
  {year:2017,fastest:'China',pct:6.1,growthB:12.75,topSpender:'United States',topB:827.67},
  {year:2018,fastest:'United States',pct:3.0,growthB:24.92,topSpender:'United States',topB:852.59},
  {year:2019,fastest:'United States',pct:5.7,growthB:48.44,topSpender:'United States',topB:901.03},
  {year:2020,fastest:'United States',pct:4.7,growthB:42.42,topSpender:'United States',topB:943.45},
  {year:2021,fastest:'China',pct:2.6,growthB:6.67,topSpender:'United States',topB:933.33},
  {year:2022,fastest:'Russia',pct:29.5,growthB:20.18,topSpender:'United States',topB:922.55},
  {year:2023,fastest:'United States',pct:2.2,growthB:20.48,topSpender:'United States',topB:943.03},
  {year:2024,fastest:'United States',pct:6.6,growthB:61.87,topSpender:'United States',topB:1004.90},
  {year:2025,fastest:'China',pct:7.4,growthB:23.12,topSpender:'United States',topB:929.16},
];
const colorMap={'United States':'#808080','China':'#e05c2a','Russia':'#3a7abf','India':'#8050b0','Saudi Arabia':'#d4a020'};
const getColor=c=>colorMap[c]||'#50a060';
const annotations={1997:'Geopolitical tensions',1999:'China WTO accession',2002:'9/11 aftermath',2009:'Afghanistan surge',2022:'Russia invades Ukraine'};
new Chart(document.getElementById('growthChart'),{
  type:'bar',
  data:{labels:rows.map(r=>r.year),datasets:[{data:rows.map(r=>r.pct),backgroundColor:rows.map(r=>getColor(r.fastest)),borderRadius:3,borderSkipped:false}]},
  options:{responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false},tooltip:{callbacks:{title:ctx=>{const r=rows[ctx[0].dataIndex];return r.year+(annotations[r.year]?'  —  '+annotations[r.year]:'');},label:ctx=>{const r=rows[ctx.dataIndex];return['Fastest grower:     '+r.fastest,'Growth rate:        +'+r.pct+'%','Absolute increase:  +$'+r.growthB.toFixed(1)+'B','──────────────────────────','Top spender:        '+r.topSpender,'Top spending:       $'+r.topB.toFixed(1)+'B'];}},backgroundColor:'rgba(10,10,10,0.95)',titleColor:'#ffffff',bodyColor:'#ffffff',padding:12,displayColors:false,borderColor:'#2a2a2a',borderWidth:1}},scales:{x:{grid:{display:false},ticks:{color:'#444',font:{size:10,family:'Courier New'},autoSkip:false,maxRotation:45,callback:(v,i)=>rows[i].year}},y:{grid:{color:'rgba(255,255,255,0.04)'},ticks:{color:'#444',font:{size:11,family:'Courier New'},callback:v=>'+'+v+'%'},title:{display:true,text:'Fastest growth rate that year (%)',color:'#444',font:{size:11,family:'Courier New'}}}}}
});
</script>
</body>
</html>'''

with open('charts/chart3_growth.html', 'w') as f:
    f.write(html_chart3)
print("✅ charts/chart3_growth.html saved")


✅ charts/chart3_growth.html saved


---
## 4. Summary — The Big Picture

| Dimension | Finding |
|-----------|---------|
| **Who dominates in absolute terms** | United States — every year, no exception |
| **Who is catching up fastest** | China — +1,583% over 36 years, now at #2 |
| **Most volatile country** | Russia — 9 ranking changes, collapsed then resurged |
| **Most reactive to external events** | Saudi Arabia — sharp increases driven by geopolitical tensions and weapons modernisation |
| **Sleeper story** | India — quiet, consistent climb from #7 to #5, now ahead of France and UK |
| **European story** | Stagnation until 2022, then Germany and UK rapidly accelerating |
| **Post-USSR collapse** | Russia's 1992 spending ($47B) was 17% of USSR's 1988 spending ($283B) |
| **Most turbulent years** | 2017 (4 events) and 2024 (4 events) |
| **Single largest growth spike** | Saudi Arabia 1997: +35.8% in one year |
| **Second largest** | Russia 2022: +29.5% — Ukraine invasion |

### What the data does NOT show
- Military capability (a dollar spent in China ≠ a dollar spent in the US due to cost differences)  
- Nuclear arsenals, personnel numbers, or technological sophistication  
- Aid and proxy spending (US arms transfers to Ukraine not fully captured here)  

### Data source
Stockholm International Peace Research Institute (SIPRI) Military Expenditure Database, 2026.  
Processed by Our World in Data. Figures in constant 2024 USD.


---
## 5. Predictive Analytics -- Military Spending Forecast (2026, 2028, 2030)

### Why three points instead of a continuous forecast
Short-term point forecasts (1-5 years) are more credible than long-range projections.
Presenting 2026, 2028, 2030 gives decision-makers three concrete reference points
while avoiding the false precision of a continuous line to 2035.

### Models used

**Model 1 -- Polynomial Regression (degree 2)**
Fits a quadratic curve to 1988-2025 historical data.
Captures long-run structural trends (acceleration or deceleration).
*Risk: sensitive to the curvature of the full historical series.*

**Model 2 -- Exponential Smoothing (alpha=0.3)**
Weights recent years more heavily. Uses the last 5 years of smoothed trend to project forward.
More responsive to recent behavioral shifts (e.g. post-Ukraine Russia, post-NATO-pressure Germany).
*Risk: anchored to recent momentum, may miss mean-reversion.*

**Rule of thumb:** When both models agree, confidence is higher. When they diverge, uncertainty is real.


In [24]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

df_raw = pd.read_csv('military-spending-sipri.csv')
df_raw = df_raw[df_raw['Code'].notna()].copy()
df_raw = df_raw.rename(columns={'Entity':'country','Year':'year','Military expenditure':'spending'})
df_raw['spending_B'] = (df_raw['spending'] / 1e9).round(2)

COUNTRIES  = ['United States','China','Russia','India','United Kingdom','Germany']
TARGET_YEARS = [2026, 2028, 2030]
ALL_FUTURE   = list(range(2026, 2031))
ALPHA = 0.3

forecast_results = {}

for c in COUNTRIES:
    cdf = df_raw[
        (df_raw['country'] == c) &
        (df_raw['year'] >= 1988) &
        (df_raw['year'] <= 2025)
    ].dropna(subset=['spending_B']).sort_values('year')

    X = cdf['year'].values.reshape(-1,1)
    y = cdf['spending_B'].values

    # Model 1: Polynomial Regression
    poly_model = make_pipeline(PolynomialFeatures(2), LinearRegression())
    poly_model.fit(X, y)
    poly_all  = np.maximum(poly_model.predict(np.array(ALL_FUTURE).reshape(-1,1)), 0)
    poly_pred = {yr: round(float(poly_all[ALL_FUTURE.index(yr)]),1) for yr in TARGET_YEARS}

    # Model 2: Exponential Smoothing
    smoothed = [y[0]]
    for i in range(1, len(y)):
        smoothed.append(ALPHA * y[i] + (1-ALPHA) * smoothed[-1])
    last_smooth  = float(smoothed[-1])
    recent_trend = float(np.mean(np.diff(smoothed[-5:])))
    exp_pred = {yr: round(max(0.0, last_smooth + recent_trend*(yr-2025)),1) for yr in TARGET_YEARS}

    forecast_results[c] = {
        'hist_years':  cdf['year'].tolist(),
        'hist_vals':   [round(float(v),1) for v in y.tolist()],
        'last_actual': round(float(y[-1]),1),
        'poly': poly_pred,
        'exp':  exp_pred,
    }

# Summary table
print(f"{'Country':<20} {'2025':>8} | {'2026 Poly':>9} {'2026 Exp':>9} | {'2028 Poly':>9} {'2028 Exp':>9} | {'2030 Poly':>9} {'2030 Exp':>9}")
print("-"*95)
for c, r in forecast_results.items():
    print(f"{c:<20} ${r['last_actual']:>6.1f}B | "
          f"  ${r['poly'][2026]:>6.1f}B   ${r['exp'][2026]:>6.1f}B | "
          f"  ${r['poly'][2028]:>6.1f}B   ${r['exp'][2028]:>6.1f}B | "
          f"  ${r['poly'][2030]:>6.1f}B   ${r['exp'][2030]:>6.1f}B")


Country                  2025 | 2026 Poly  2026 Exp | 2028 Poly  2028 Exp | 2030 Poly  2030 Exp
-----------------------------------------------------------------------------------------------
United States        $ 929.2B |   $1007.8B   $ 949.6B |   $1031.7B   $ 966.8B |   $1056.2B   $ 984.1B
China                $ 335.0B |   $ 353.5B   $ 308.4B |   $ 390.3B   $ 336.5B |   $ 429.1B   $ 364.5B
Russia               $ 158.2B |   $ 136.2B   $ 133.3B |   $ 153.3B   $ 159.8B |   $ 171.7B   $ 186.3B
India                $  93.2B |   $  97.0B   $  87.6B |   $ 103.8B   $  92.4B |   $ 110.9B   $  97.2B
United Kingdom       $  88.0B |   $  82.4B   $  83.9B |   $  85.0B   $  89.3B |   $  87.8B   $  94.8B
Germany              $ 106.7B |   $  86.0B   $  86.4B |   $  94.8B   $  98.6B |   $ 104.5B   $ 110.8B


### Key Forecast Insights

#### United States: $950B-$1,056B by 2030
- Polynomial model crosses **$1 trillion by 2026** -- driven by the 2002-2009 war spending curve
- Exponential smoothing is more conservative ($984B by 2030) -- anchored to recent moderate growth
- **Consensus: US will be in the $950B-$1,056B range by 2030**, likely crossing $1T before then
- Remains #1 by a factor of 2.4x-2.9x over China

#### China: $308B-$429B by 2030
- **Largest model divergence of all 6 countries** -- $121B gap by 2030
- Polynomial captures China's historic steep curve; exponential smoothing reflects the recent growth slowdown
- Exponential model ($364B) is likely more realistic given visible deceleration since 2020
- **Key question:** Does China re-accelerate or continue slowing? This single decision drives $100B+ of uncertainty

#### Russia: $133B-$186B by 2030 -- the most interesting divergence
- In 2026, both models nearly agree (~$133-136B) -- a **downward projection from 2025's $158B**
- This means both models expect Russia's war-level spending to be **unsustainable**
- By 2030, exponential smoothing projects $186B (war momentum continues) vs polynomial $172B
- **The models are telling you Russia peaked in 2022-2025 -- watch this carefully**

#### India: $88B-$111B by 2030
- Highest model agreement -- steady, predictable trajectory
- Projected to stay in the **$88B-$111B range** -- consistent with historic 5-8% annual growth
- On track to definitively pass UK by 2028-2030 on both models

#### Germany: $86B-$111B by 2030 -- a structural break story
- Both models show 2026 **below 2025** ($86B vs $107B actual)
- This is because the models are anchored to pre-2024 data and the recent NATO surge looks like an outlier
- If Germany sustains 2% GDP commitment, actual will **exceed both projections significantly**
- This is a case where the model is likely wrong because of a known structural break

#### United Kingdom: $82B-$95B by 2030
- Polynomial projects **decline from 2025** ($88B -> $82B by 2026)
- Exponential smoothing projects modest growth ($95B by 2030)
- Both agree UK is the weakest trajectory -- likely to be overtaken by India before 2028

---

### Model Divergence Summary
| Country | 2030 Poly | 2030 Exp | Gap | Confidence |
|---------|-----------|----------|-----|------------|
| United States | $1,056B | $984B | $72B | Medium |
| China | $429B | $365B | $121B | Low |
| Russia | $172B | $186B | $14B | High (but structurally uncertain) |
| India | $111B | $97B | $14B | High |
| Germany | $105B | $111B | $6B | Medium (structural break likely) |
| United Kingdom | $88B | $95B | $7B | Medium |


In [25]:
# Divergence analysis
print("Model divergence at 2030:")
print()
for c, r in forecast_results.items():
    gap     = abs(r['poly'][2030] - r['exp'][2030])
    pct_gap = gap / r['last_actual'] * 100
    level   = 'HIGH agreement' if pct_gap < 10 else 'MEDIUM agreement' if pct_gap < 20 else 'LOW agreement'
    direction = 'GROWTH' if min(r['poly'][2030], r['exp'][2030]) > r['last_actual'] else 'DECLINE/FLAT'
    print(f"  {c:<20}  gap=${gap:>5.1f}B ({pct_gap:>4.0f}% of 2025)  {level:<20}  trend: {direction}")


Model divergence at 2030:

  United States         gap=$ 72.1B (   8% of 2025)  HIGH agreement        trend: GROWTH
  China                 gap=$ 64.6B (  19% of 2025)  MEDIUM agreement      trend: GROWTH
  Russia                gap=$ 14.6B (   9% of 2025)  HIGH agreement        trend: GROWTH
  India                 gap=$ 13.7B (  15% of 2025)  MEDIUM agreement      trend: GROWTH
  United Kingdom        gap=$  7.0B (   8% of 2025)  HIGH agreement        trend: DECLINE/FLAT
  Germany               gap=$  6.3B (   6% of 2025)  HIGH agreement        trend: DECLINE/FLAT


### Generate Chart 4 -- Forecast Comparison (Interactive HTML)

In [26]:
import json

chart_data   = {"United States": {"hist_years": [1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025], "hist_vals": [821.4, 814.5, 780.6, 689.5, 726.6, 687.8, 652.0, 609.0, 575.9, 572.9, 560.0, 561.4, 583.1, 587.8, 660.0, 751.2, 818.8, 856.5, 868.8, 892.0, 956.9, 1032.2, 1061.7, 1049.1, 990.8, 914.6, 858.4, 838.9, 836.3, 827.7, 852.6, 901.0, 943.5, 933.3, 922.5, 943.0, 1004.9, 929.2], "last_actual": 929.2, "poly": {"2026": 1007.8, "2028": 1031.7, "2030": 1056.2}, "exp": {"2026": 949.6, "2028": 966.8, "2030": 984.1}}, "China": {"hist_years": [1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025], "hist_vals": [19.9, 21.6, 23.0, 28.0, 25.7, 24.8, 25.7, 27.2, 29.1, 31.7, 38.7, 41.9, 49.6, 57.0, 61.7, 68.0, 74.6, 85.9, 94.4, 103.3, 125.4, 131.5, 141.2, 155.7, 168.4, 181.8, 196.1, 207.4, 220.1, 233.0, 244.4, 255.9, 262.6, 274.2, 292.2, 311.9, 335.0], "last_actual": 335.0, "poly": {"2026": 353.5, "2028": 390.3, "2030": 429.1}, "exp": {"2026": 308.4, "2028": 336.5, "2030": 364.5}}, "Russia": {"hist_years": [1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025], "hist_vals": [47.5, 41.5, 39.2, 25.8, 24.4, 26.6, 15.8, 17.6, 23.7, 25.6, 28.4, 29.8, 31.1, 35.4, 39.2, 42.6, 46.9, 49.2, 50.2, 53.5, 62.0, 65.0, 69.7, 75.1, 80.5, 65.3, 62.8, 65.6, 67.2, 68.5, 88.7, 109.0, 149.4, 158.2], "last_actual": 158.2, "poly": {"2026": 136.2, "2028": 153.3, "2030": 171.7}, "exp": {"2026": 133.3, "2028": 159.8, "2030": 186.3}}, "India": {"hist_years": [1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025], "hist_vals": [21.1, 22.2, 21.9, 20.4, 19.5, 22.0, 22.1, 22.8, 23.2, 25.7, 26.8, 31.1, 32.1, 33.3, 33.2, 33.9, 39.4, 41.9, 42.2, 42.8, 48.5, 57.1, 57.3, 57.9, 57.6, 57.6, 60.4, 61.0, 67.2, 71.9, 74.5, 79.7, 80.2, 79.7, 83.2, 85.2, 85.6, 93.2], "last_actual": 93.2, "poly": {"2026": 97.0, "2028": 103.8, "2030": 110.9}, "exp": {"2026": 87.6, "2028": 92.4, "2030": 97.2}}, "United Kingdom": {"hist_years": [1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025], "hist_vals": [74.4, 75.1, 75.2, 76.1, 71.0, 67.8, 65.9, 61.8, 61.2, 59.1, 59.3, 59.0, 60.1, 62.5, 66.2, 71.0, 71.8, 72.5, 72.8, 74.8, 78.1, 79.3, 78.0, 75.4, 73.3, 70.6, 69.3, 66.7, 66.4, 66.6, 67.1, 69.8, 71.0, 72.0, 73.2, 78.2, 89.8, 88.0], "last_actual": 88.0, "poly": {"2026": 82.4, "2028": 85.0, "2030": 87.8}, "exp": {"2026": 83.9, "2028": 89.3, "2030": 94.8}}, "Germany": {"hist_years": [1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025], "hist_vals": [67.2, 67.0, 70.6, 65.3, 62.2, 55.9, 52.1, 51.2, 50.2, 48.4, 48.6, 49.5, 48.8, 48.0, 48.1, 47.4, 46.0, 45.3, 44.5, 44.5, 45.6, 47.3, 47.4, 46.5, 47.8, 46.0, 46.1, 46.9, 48.9, 50.3, 51.7, 56.7, 60.4, 59.9, 62.5, 70.2, 86.2, 106.7], "last_actual": 106.7, "poly": {"2026": 86.0, "2028": 94.8, "2030": 104.5}, "exp": {"2026": 86.4, "2028": 98.6, "2030": 110.8}}}
country_colors = {"United States": "#808080", "China": "#e05c2a", "Russia": "#3a7abf", "India": "#8050b0", "United Kingdom": "#50a060", "Germany": "#d4a020"}
chart_data_json = json.dumps(chart_data)
colors_json = json.dumps(country_colors)

html = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Military Spending Forecast 2026-2030</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
<style>
  *{margin:0;padding:0;box-sizing:border-box;}
  body{background:#0d0d0d;font-family:Georgia,serif;color:#E8E4DC;min-height:100vh;display:flex;align-items:center;justify-content:center;padding:40px 24px;}
  .card{background:white;border:0.5px solid #2a2a2a;border-radius:12px;padding:36px 40px;width:100%;max-width:980px;}
  .header{margin-bottom:20px;}
  .header h1{font-size:18px;font-weight:normal;letter-spacing:0.04em;color:black;margin-bottom:6px;}
  .header p{font-size:12px;color:black;font-family:'Courier New',monospace;}
  .controls{display:flex;flex-wrap:wrap;gap:10px;margin-bottom:16px;}
  .btn{padding:5px 14px;border-radius:20px;border:0.5px solid #2a2a2a;background:white;color:black;font-size:11px;font-family:'Courier New',monospace;cursor:pointer;transition:all 0.15s;}
  .btn:hover,.btn.active{background:#f0f0f0;color:black;border-color:#555;}
  .legend{display:flex;flex-wrap:wrap;gap:14px;margin-bottom:16px;font-size:11px;color:black;font-family:'Courier New',monospace;}
  .legend-item{display:flex;align-items:center;gap:5px;}
  .legend-dot{width:9px;height:9px;border-radius:50%;flex-shrink:0;}
  .chart-wrap{position:relative;width:100%;height:400px;}
  .table-wrap{overflow-x:auto;margin-top:24px;}
  table{width:100%;border-collapse:collapse;font-size:11px;font-family:'Courier New',monospace;}
  th{padding:8px 12px;text-align:right;color:black;border-bottom:0.5px solid #2a2a2a;font-weight:normal;}
  th:first-child{text-align:left;}
  td{padding:7px 12px;text-align:right;color:black;border-bottom:0.5px solid #e8e8e8;}
  td:first-child{text-align:left;}
  .poly{color:#7ab0e0;}
  .exp{color:#e0b070;}
  .actual{color:black;font-weight:bold;}
  .divider{width:100%;height:0.5px;background:#1e1e1e;margin:20px 0;}
  .notes{display:flex;flex-direction:column;gap:7px;}
  .note{display:flex;gap:10px;font-size:11px;color:black;font-family:'Courier New',monospace;line-height:1.5;}
  .note-num{color:#3a3a3a;flex-shrink:0;}
  .note em{color:black;font-style:normal;}
  .source{margin-top:20px;padding-top:16px;border-top:0.5px solid #1e1e1e;font-size:10px;color:#3a3a3a;font-family:'Courier New',monospace;}
</style>
</head>
<body>
<div class="card">
  <div class="header">
    <h1>Military spending forecast -- 2026, 2028, 2030</h1>
    <p>Historical: SIPRI 1988-2025 (solid) -- Forecast: Polynomial Regression (dashed) vs Exponential Smoothing (dotted)</p>
  </div>

  <div class="controls">
    <span style="font-size:11px;color:black;font-family:'Courier New',monospace;align-self:center;">Show model:</span>
    <button class="btn active" id="btnBoth" onclick="setModel('both')">Both models</button>
    <button class="btn" id="btnPoly" onclick="setModel('poly')">Polynomial only</button>
    <button class="btn" id="btnExp"  onclick="setModel('exp')">Exp. Smoothing only</button>
  </div>

  <div class="legend" id="legend"></div>

  <div class="chart-wrap">
    <canvas id="forecastChart" role="img" aria-label="Line chart of military spending history and forecast for 6 countries.">Military spending forecast.</canvas>
  </div>

  <div class="divider"></div>

  <div class="table-wrap">
    <table>
      <thead>
        <tr>
          <th>Country</th>
          <th>2025 actual</th>
          <th class="poly">2026 Poly</th><th class="exp">2026 Exp</th>
          <th class="poly">2028 Poly</th><th class="exp">2028 Exp</th>
          <th class="poly">2030 Poly</th><th class="exp">2030 Exp</th>
        </tr>
      </thead>
      <tbody id="forecastTable"></tbody>
    </table>
  </div>

  <div class="divider"></div>
  <div class="notes">
    <div class="note"><span class="note-num">1</span><span><em>Solid lines</em> = SIPRI historical data. <em>Dashed</em> = Polynomial Regression. <em>Dotted</em> = Exponential Smoothing. Hover for values.</span></div>
    <div class="note"><span class="note-num">2</span><span>Large gap between models = high uncertainty. China ($121B gap by 2030) and US ($72B gap) carry the most uncertainty.</span></div>
    <div class="note"><span class="note-num">3</span><span><em>Germany and Russia projections require caution:</em> Germany's 2024-2025 NATO surge may not be captured by models trained on pre-surge data. Russia's war spending may not be sustainable.</span></div>
  </div>
  <div class="source">Models: Polynomial Regression (sklearn, degree 2) and Exponential Smoothing (alpha=0.3, 5-year trend) -- Data: SIPRI 2026 via Our World in Data</div>
</div>

<script>
const chartData    = """ + chart_data_json + """;
const countryColors = """ + colors_json + """;
const countries = Object.keys(chartData);
const targetYears = [2026, 2028, 2030];
let currentModel = 'both';
let chart;

function buildDatasets() {
  const datasets = [];
  countries.forEach(c => {
    const d = chartData[c];
    const col = countryColors[c];
    const histData = d.hist_years.map((y,i) => ({x:y, y:d.hist_vals[i]}));
    datasets.push({label:c+' (actual)', data:histData, borderColor:col, backgroundColor:'transparent', borderWidth:2, pointRadius:0, tension:0.3});
    if (currentModel==='both'||currentModel==='poly') {
      const polyData = targetYears.map(yr => ({x:yr, y:d.poly[yr]}));
      const bridgePoint = {x: d.hist_years[d.hist_years.length-1], y: d.hist_vals[d.hist_vals.length-1]};
      datasets.push({label:c+' (poly)', data:[bridgePoint,...polyData], borderColor:col, backgroundColor:'transparent', borderWidth:1.5, borderDash:[6,3], pointRadius:[0,5,5,5], pointBackgroundColor:col, tension:0.2});
    }
    if (currentModel==='both'||currentModel==='exp') {
      const expData = targetYears.map(yr => ({x:yr, y:d.exp[yr]}));
      const bridgePoint = {x: d.hist_years[d.hist_years.length-1], y: d.hist_vals[d.hist_vals.length-1]};
      datasets.push({label:c+' (exp)', data:[bridgePoint,...expData], borderColor:col, backgroundColor:'transparent', borderWidth:1.5, borderDash:[2,4], pointRadius:[0,5,5,5], pointBackgroundColor:col, tension:0.2});
    }
  });
  return datasets;
}

function buildLegend() {
  const leg = document.getElementById('legend');
  leg.innerHTML = '';
  countries.forEach(c => {
    const col = countryColors[c];
    const item = document.createElement('div');
    item.className = 'legend-item';
    item.innerHTML = '<div class="legend-dot" style="background:'+col+'"></div>'+c;
    leg.appendChild(item);
  });
  const note = document.createElement('div');
  note.style.cssText = 'width:100%;font-size:10px;color:#3a3a3a;font-family:Courier New,monospace;margin-top:2px;';
  note.textContent = '-- solid: historical   - - dashed: polynomial   ... dotted: exp. smoothing   dots: forecast points';
  leg.appendChild(note);
}

function buildTable() {
  const tbody = document.getElementById('forecastTable');
  tbody.innerHTML = '';
  countries.forEach(c => {
    const d = chartData[c];
    const col = countryColors[c];
    const tr = document.createElement('tr');
    tr.innerHTML =
      '<td style="color:'+col+'">'+c+'</td>'+
      '<td class="actual">$'+d.last_actual+'B</td>'+
      targetYears.map(yr =>
        '<td class="poly">$'+d.poly[yr]+'B</td><td class="exp">$'+d.exp[yr]+'B</td>'
      ).join('');
    tbody.appendChild(tr);
  });
}

function setModel(model) {
  currentModel = model;
  ['btnBoth','btnPoly','btnExp'].forEach(id => document.getElementById(id).classList.remove('active'));
  const btnId = model==='both'?'btnBoth':model==='poly'?'btnPoly':'btnExp';
  document.getElementById(btnId).classList.add('active');
  chart.data.datasets = buildDatasets();
  chart.update();
}

function drawForecastZone() {
  const ctx2 = chart.ctx;
  const xScale = chart.scales.x;
  const x2025 = xScale.getPixelForValue(2025.3);
  ctx2.save();
  ctx2.fillStyle = 'rgba(255,255,255,0.02)';
  ctx2.fillRect(x2025, chart.chartArea.top, chart.chartArea.right-x2025, chart.chartArea.bottom-chart.chartArea.top);
  ctx2.setLineDash([4,4]);
  ctx2.strokeStyle = 'rgba(255,255,255,0.15)';
  ctx2.lineWidth = 1;
  ctx2.beginPath();
  ctx2.moveTo(x2025, chart.chartArea.top);
  ctx2.lineTo(x2025, chart.chartArea.bottom);
  ctx2.stroke();
  ctx2.setLineDash([]);
  ctx2.fillStyle = 'rgba(255,255,255,0.2)';
  ctx2.font = '10px Courier New';
  ctx2.fillText('forecast zone ->', x2025+8, chart.chartArea.top+14);
  ctx2.restore();
}

window.onload = function() {
  buildLegend();
  buildTable();
  chart = new Chart(document.getElementById('forecastChart'), {
    type:'line',
    data:{datasets:buildDatasets()},
    options:{
      responsive:true, maintainAspectRatio:false, parsing:false,
      plugins:{
        legend:{display:false},
        tooltip:{
          mode:'nearest', intersect:true,
          callbacks:{
            title:ctx => 'Year: '+ctx[0].raw.x,
            label:ctx => {
              const parts = ctx.dataset.label.split(' (');
              const country = parts[0];
              const type = (parts[1]||'actual)').replace(')','');
              const typeLabel = type==='actual'?'actual':type==='poly'?'Poly forecast':'Exp forecast';
              return country+' ['+typeLabel+']: $'+ctx.raw.y.toFixed(1)+'B';
            }
          },
          backgroundColor:'rgba(10,10,10,0.95)',titleColor:'#e8e4dc',bodyColor:'#888',
          padding:12,displayColors:false,borderColor:'#2a2a2a',borderWidth:1,
        }
      },
      scales:{
        x:{type:'linear',min:1995,max:2031,
           grid:{color:'rgba(255,255,255,0.04)'},
           ticks:{color:'#444',font:{size:10,family:'Courier New'},stepSize:5,callback:v=>Number.isInteger(v)?v:''}},
        y:{grid:{color:'rgba(255,255,255,0.04)'},
           ticks:{color:'#444',font:{size:11,family:'Courier New'},callback:v=>'$'+v+'B'},
           title:{display:true,text:'Billion USD (constant 2024)',color:'#444',font:{size:11,family:'Courier New'}}}
      },
      animation:{onComplete:()=>drawForecastZone()},
    }
  });
};
</script>
</body>
</html>"""

with open('charts/chart4_forecast.html','w') as f:
    f.write(html)
print("charts/chart4_forecast.html saved")


charts/chart4_forecast.html saved


### Model Limitations and Professional Caveats

| Limitation | Country most affected |
|------------|----------------------|
| Structural break not captured | Germany (NATO 2% surge), Russia (war spending) |
| Short post-break history | Germany, UK |
| War spending unsustainability | Russia |
| Growth deceleration vs historic curve | China |
| Political budget cycles | United States |

**What a full professional model would add:**
- Scenario modeling with GDP growth assumptions (optimistic / base / pessimistic)
- Monte Carlo confidence bands (e.g. 80% prediction interval)
- Exogenous variable regression (oil price x Saudi Arabia, GDP x China)
- Cross-validation: train on 1988-2015, test on 2016-2025, measure error before forecasting
